# CCD height map: sign convention and the Z4 it adds (v1)

**Author:** Aaron Roodman (prepared by Claude)  
**Date Created:** 2026-10-24  
**Last Modified:** 2026-10-24  
**Status:** In Progress  
**Keywords:** CCD height map, Z4, defocus, corner wavefront sensor, batoid, ts_wep, metrology  

## Description

Checks, from the raw metrology and from batoid, which way a positive value in the CCD height
map points, and the sign of the Z4 a height adds. The test fact: on the corner wavefront
sensors (CWFS), SW1 is intra-focal (high) and SW0 extra-focal (low), at ±1.5 mm.

1. The raw metrology pistons of SW0 and SW1, so the sign of "intra" in the map's frame is read
   off the data.
2. The batoid frame: which way the Detector's +z points, and the Z4 of a ±1.5 mm shift (ts_wep's
   intra/extra convention).
3. The batoid_rubin height map against the metrology, after the ±1.5 mm piston is removed.
4. The Z4 batoid gets from the height map on each corner sensor, against the
   `ts_intrinsic_wavefront` term `+15 µm/mm × h`.

**Output:** printed tables only  
**References:** `notes/status/phase6_ccd_height_sign.md`; `aos/docs/ts_wep_zernike_intrinsics.md` §3;
`ts_intrinsic_wavefront` `ccd_height.py`

## Parameters

In [1]:
import os

# Raw SLAC metrology (per-sensor BinTableHDUs with X_CCS, Y_CCS, Z_CCS_MODEL, Z_CCS_MEASURED)
METROLOGY_FITS = '/sdf/home/g/gmegias/aos/chip_heights/height_map/LSST_FP_cold_b_measurement_4col_bysurface.fits.gz'
# batoid_rubin per-detector height map (ccd_height_map.fits.gz)
HEIGHT_MAP_DIR = os.path.expanduser('~/u/LSST/packages/batoid_rubin_data/ccd_height_map')
BAND = 'r'
CORNERS = ['R00', 'R04', 'R40', 'R44']

## Setup & Imports

In [2]:
import numpy as np
import pandas as pd
import batoid
from astropy.io import fits
from batoid_rubin.builder import det_height_maps
from lsst.ts.wep.instrument import Instrument
from lsst.ts.intrinsic.wavefront import ccd_height as ch

inst = Instrument(configFile='policy:instruments/LsstCam.yaml')
tel = inst.getBatoidModel(BAND)
wl = inst.wavelength[BAND]
eps = tel.pupilObscuration
maps = det_height_maps(HEIGHT_MAP_DIR)
print(f'band {BAND}, wavelength {wl*1e9:.0f} nm, obscuration {eps:.3f} (dimensionless), '
      f'defocalOffset {inst.defocalOffset*1e3:.2f} mm')

def z4_ta(optic, thx, thy):
    """Z4 [µm of wavefront] from batoid.zernikeTA (annular Noll, ts_wep's estimator)."""
    return batoid.zernikeTA(optic, thx, thy, wl, eps=eps, jmax=4, nrad=20, naz=120)[4] * wl * 1e6

band r, wavelength 619 nm, obscuration 0.612 (dimensionless), defocalOffset 1.50 mm


## 1. Raw metrology: which sign is intra-focal?

Median `Z_CCS_MEASURED` per corner-sensor half, as stored in the file (mm). No piston removed.

In [3]:
rows = []
with fits.open(METROLOGY_FITS) as h:
    for hdu in h[1:]:
        if 'WFS' in hdu.name or hdu.name == 'R22S11':
            rows.append((hdu.name, float(np.median(hdu.data['Z_CCS_MEASURED']))))
met = pd.DataFrame(rows, columns=['sensor', 'median Z_CCS_MEASURED [mm]'])
met

,sensor,median Z_CCS_MEASURED [mm]
0,R00WFS0,1.499184
1,R00WFS1,-1.502338
2,R04WFS0,1.501678
3,R04WFS1,-1.501853
4,R22S11,0.005289
5,R40WFS0,1.502409
6,R40WFS1,-1.508552
7,R44WFS0,1.496810
8,R44WFS1,-1.499044


WFS0 (= SW0, extra-focal) sits at +1.5 mm and WFS1 (= SW1, intra-focal, high) at −1.5 mm.
**So in this file a negative value is toward intra-focal (high), and a positive value toward
extra-focal (low).**

## 2. The batoid frame and ts_wep's intra/extra Z4

Which way the Detector's local +z points relative to the light, and Z4 at a field point for the ts_wep defocal shifts.

In [4]:
det = tel['Detector']
r = batoid.RayVector.fromStop(0, 0, tel, wavelength=wl, theta_x=0, theta_y=np.deg2rad(0.5))
tel.trace(r)
print(f'Detector at global z {det.coordSys.origin[2]:.5f} m, local zhat {det.coordSys.zhat}; '
      f'L3 exit at global z {tel["L3_exit"].coordSys.origin[2]:.5f} m')
print(f'chief ray direction at the detector (global): vz = {r.vz[0]:+.4f} (light travels +z)')
for sign, label in [(+1, 'extra (ts_wep sign +1)'), (-1, 'intra (ts_wep sign -1)')]:
    shifted = tel.withLocallyShiftedOptic('Detector', [0, 0, sign * inst.defocalOffset])
    print(f'{label:24s}: Detector shifted {sign*inst.defocalOffset*1e3:+.1f} mm, '
          f'Z4 = {z4_ta(shifted, 0, np.deg2rad(0.5)):+.3f} µm of wavefront')

Detector at global z 4.42942 m, local zhat [0. 0. 1.]; L3 exit at global z 4.40092 m
chief ray direction at the detector (global): vz = +0.9994 (light travels +z)
extra (ts_wep sign +1)  : Detector shifted +1.5 mm, Z4 = +23.797 µm of wavefront
intra (ts_wep sign -1)  : Detector shifted -1.5 mm, Z4 = -23.766 µm of wavefront


Light travels +z, so +z is away from L3. ts_wep's extra-focal shift is +1.5 mm (Z4 > 0); intra
is −1.5 mm, toward L3 (high), with Z4 < 0. This matches the metrology: SW0 at +1.5 mm, SW1 at
−1.5 mm. **An intra-focal (high) displacement gives negative Z4.**

## 3. The batoid_rubin height map against the metrology

`ts_intrinsic_wavefront.ccd_height.make_metrology_table` removes the ±1.5 mm piston from the WFS halves. Compare the result with the batoid map at the same points.

In [5]:
t = ch.make_metrology_table(METROLOGY_FITS)
dets = np.asarray(t['det']).astype(str)
keep = np.isin(dets, [f'{c}_{s}' for c in CORNERS for s in ('SW0', 'SW1')] + ['R22_S11', 'R11_S11', 'R33_S22'])
fx, fy, zm, dd = (np.asarray(t['fpx'])[keep], np.asarray(t['fpy'])[keep],
                  np.asarray(t['z_meas'])[keep], dets[keep])
hb = ch._sag_oriented(maps, fx, fy, dd, *ch.BATOID_FP_ORIENTATION)
ok = np.isfinite(hb)
slope = np.polyfit(zm[ok], hb[ok], 1)[0]
print(f'{ok.sum()} points (count): slope batoid/metrology = {slope:+.3f} (dimensionless); '
      f'RMS(batoid - metrology) = {np.std(hb[ok]-zm[ok])*1e3:.2f} µm, '
      f'RMS(batoid + metrology) = {np.std(hb[ok]+zm[ok])*1e3:.2f} µm')
pd.DataFrame([(d, np.median(zm[ok & (dd == d)])*1e3, np.median(hb[ok & (dd == d)])*1e3)
              for d in np.unique(dd)],
             columns=['detector', 'metrology h [µm]', 'batoid map h [µm]'])

17391 points (count): slope batoid/metrology = +0.949 (dimensionless); RMS(batoid - metrology) = 0.95 µm, RMS(batoid + metrology) = 8.37 µm


,detector,metrology h [µm],batoid map h [µm]
0,R00_SW0,-0.815988,-0.733896
1,R00_SW1,-2.338052,-2.510238
2,R04_SW0,1.677990,1.720926
3,R04_SW1,-1.852989,-1.772007
4,R11_S11,-0.195000,-0.244078
5,R22_S11,5.289500,5.342823
6,R33_S22,-1.003000,-1.035591
7,R40_SW0,2.408981,2.359478
8,R40_SW1,-8.552015,-8.577557
9,R44_SW0,-3.189981,-3.308725


The batoid map has the metrology's sign (slope +0.95; flipping it raises the RMS from about
1 µm to about 8 µm). **So the height map inherits the metrology convention: positive h is
toward extra-focal (low), negative h toward intra-focal (high).** batoid applies it with
`withPerturbedSurface('Detector', ...)`, as a sag along the Detector's +z, the same direction.

## 4. The Z4 a height adds, per corner sensor

For each corner sensor, a field point whose chief ray lands at the map center; SW0 traced extra (+1.5 mm), SW1 intra (−1.5 mm). dZ4 (batoid) = Z4 with the map minus without. dZ4 (pipeline) = `+15 µm/mm × h`, what `compute_ccd_heights` returns as `Z4_height`.

In [6]:
def land(optic, th):
    r = batoid.RayVector.fromStop(0, 0, optic, wavelength=wl, theta_x=th[0], theta_y=th[1])
    optic.trace(r)
    return np.array([r.x[0], r.y[0]])

rows = []
for raft in CORNERS:
    for side, sw, sign in [('extra', 'SW0', +1), ('intra', 'SW1', -1)]:
        m = maps[f'{raft}_{sw}']
        target = np.array([0.5*(m.xs[0]+m.xs[-1]), 0.5*(m.ys[0]+m.ys[-1])])
        shifted = tel.withLocallyShiftedOptic('Detector', [0, 0, sign*inst.defocalOffset])
        th = target / 10.31                   # plate scale ~10.31 m/rad, then Newton steps
        for _ in range(8):
            p = land(shifted, th)
            if np.hypot(*(p - target)) < 1e-7:
                break
            J = np.column_stack([(land(shifted, th + d) - p) / 1e-6 for d in np.eye(2)*1e-6])
            th = th + np.linalg.solve(J, target - p)
        h_mm = float(m.sag(*land(shifted, th))) * 1e3
        withmap = tel.withPerturbedSurface('Detector', m).withLocallyShiftedOptic(
            'Detector', [0, 0, sign*inst.defocalOffset])
        dz_b = (z4_ta(withmap, *th) - z4_ta(shifted, *th)) * 1e3
        dz_p = ch.HEIGHT_TO_Z4_UM_PER_MM * h_mm * 1e3
        rows.append((raft, sw, side, h_mm*1e3, dz_b, dz_p, dz_b/dz_p))
res = pd.DataFrame(rows, columns=['raft', 'sensor', 'side', 'h [µm]', 'dZ4 batoid [nm]',
                                  'dZ4 pipeline [nm]', 'batoid/pipeline'])
res

,raft,sensor,side,h [µm],dZ4 batoid [nm],dZ4 pipeline [nm],batoid/pipeline
0,R00,SW0,extra,-4.362393,-69.680151,-65.435897,1.064861
1,R00,SW1,intra,-4.308052,-68.609228,-64.620776,1.061721
2,R04,SW0,extra,-1.899075,-30.357587,-28.486132,1.065697
3,R04,SW1,intra,-5.451428,-86.827615,-81.771421,1.061833
4,R40,SW0,extra,-0.777891,-12.429319,-11.668372,1.065214
5,R40,SW1,intra,-11.678024,-186.069137,-175.170363,1.062218
6,R44,SW0,extra,-6.490471,-103.643008,-97.357059,1.064566
7,R44,SW1,intra,-1.300410,-20.739245,-19.506147,1.063216


## Result

- A **negative** map value is a **high** (intra-ward) point: SW1, intra-focal, is −1.5 mm in the
  raw metrology, and the map carries the metrology sign.
- An intra-ward displacement gives **negative** Z4 (ts_wep: −1.5 mm gives Z4 ≈ −23.8 µm).
- So a high point, h < 0, should add negative Z4. `Z4_height = +15 µm/mm × h` does that, and
  batoid traced with the map agrees in sign on all 8 corner sensors, intra and extra alike,
  with batoid 5–6% larger in size.
- **`ts_intrinsic_wavefront` has the right sign.** The statement to fix is in our doc: the
  term is not "extra +, intra −"; it has one sign on both sides, set by the direction of the
  height, not by which side of focus the donut is on.